# JHPC-quantum Quantum and Supercomputing Collaboration Platform
Qiskit User Guide

## 1. Introduction
This guide explains how to run quantum applications on the JHPC-quantum collaboration platform using Qiskit from the supercomputer Fugaku.

If you want to view the documentation in a browser, please see the following: 
- [jhpc-quantum GitHub](https://github.com/jhpc-quantum)
- [Release notes](https://github.com/jhpc-quantum/documents/blob/main/releasenotes.md)

## 2. What is Qiskit?
Qiskit is a collection of software tools for executing programs on quantum computers.
The Qiskit SDK is a software development kit for quantum circuits and operations on quantum computers. It is open-source software developed by IBM.
For more information, see the official Qiskit website: https://www.ibm.com/quantum/qiskit

## 3. How to Use
### 3.1. Environment Setup
If the environment is already set up, there is no need to rebuild it.
The steps below are common to both the front-end server and compute nodes (Arm) of Fugaku.

#### 3.1.1. Create a Work Directory
Create a Qiskit directory in your working directory: 

```bash
mkdir -p ${WORK}/Qiskit
cd ${WORK}/Qiskit
```

#### 3.1.2. Build a Python Virtual Environment and Install Packages
Use the environment setup script `venv_setup.sh` to create a Python virtual environment and install the required packages.
The packages include Qiskit and related packages, and the JHPC-quantum extension package for Qiskit, `qiskit-sqc-runtime`.

Copy the scripts from the shared directory: 

```bash
cp -p /vol0300/share/ra010014/jhpcq_modules/<ARCH>/SDK_latest/<Qiskit_x.x.x>/venv_setup.sh .
cp -p /vol0300/share/ra010014/jhpcq_modules/<ARCH>/SDK_latest/<Qiskit_x.x.x>/config.sh .
```

Run the environment setup script: 

```bash
./venv_setup.sh
```

### 3.2. Execution
#### 3.2.1. Environment Settings
The `backend_setup.sh` script loads the packages and environment settings required to run Qiskit programs.

```bash
#!/bin/bash
if [ "$#" -eq 1 ]; then
  QPU="$1"
  if [ "$QPU" = "reimei" ]; then
    export SQC_RPC_SERVER=ip:port
  elif [ "$QPU" = "reimei-simulator" ]; then
    export SQC_RPC_SERVER=ip:port
  elif [ "$QPU" = "ibm-kobe-dacc" ]; then
    export SQC_RPC_SERVER=ip:port
  else
    echo "invalid qpu $QPU"
  fi
else
  echo "usage: /path/to/backend_setup.sh <target-qpu-name>"
fi

source /path/to/config.sh
source /vol0004/apps/oss/spack/share/spack/setup-env.sh
spack load ${SPACK_PKG}
export LD_LIBRARY_PATH=${SQC_DIR}/lib:${SQC_DIR}/lib64:$LD_LIBRARY_PATH
export PKG_CONFIG_PATH=${SQC_DIR}/lib64/pkgconfig
export PATH=${SQC_DIR}/bin:$PATH
export SQC_LIBRARY_PATH=${SQC_DIR}/lib64
```

#### 3.2.2. Run a Sample Program
A sample program executes a two-qubit circuit on the JHPC-quantum platform:

```python
from qiskit.circuit import QuantumCircuit
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2

qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

service = QiskitRuntimeService(channel="jhpc_quantum")
backend = service.backend(name="qtm_sim_grpc")

from qiskit.transpiler import generate_preset_pass_manager
pm = generate_preset_pass_manager(target=backend.target, optimization_level=1)
isa_circuit = pm.run(qc)

sampler = SamplerV2(mode=backend)
job = sampler.run([isa_circuit], shots=10, mode="sync")
result = job.result()
print(result[0].data.meas.get_counts())
print(job.status())
```

Then run the script that invokes the sample program: 

```bash
source ./backend_setup.sh reimei-simulator
source /path/to/${VENV_NAME}/bin/activate
python ./sample.py
```

#### 3.2.3. Example Execution Results
Synchronous execution example: 

```text
{'11': 6, '00': 4}
JobStatus.DONE
```

Asynchronous execution example: 

```text
{'11': 6, '00': 4}
[['9c1406bb-c03a-4c20-8ea4-7fe62480bdb8', <JobStatus.DONE: 'job has successfully run'>]]
```

### 3.3. FireOpal Error Suppression Library
To install FireOpal, run: 

```bash
./venv_setup.sh --enable FireOpal
```

## 4. Known Issues and Workarounds
### 4.1. Process Does Not Exit on the JHPC-quantum Platform
When using Qiskit on the JHPC-quantum collaboration platform, the result may be returned but the process does not exit.
Temporary workarounds: 
- Terminate the job forcibly after the result is returned using the job limit or Ctrl+C.
- Set one of the following environment variables: 
  - export OPENBLAS_NUM_THREADS=1
  - export OMP_NUM_THREADS=1

### 4.2. Process Does Not Exit with FireOpal
When Fire Opal is used on the JHPC-quantum collaboration platform, the result may be returned but the process may not exit.
Temporary workarounds: 
- Force-stop the job after the result is returned using the job limit or Ctrl+C.
- Set one of the following environment variables: 
  - export OPENBLAS_NUM_THREADS=1
  - export OMP_NUM_THREADS=1
- Add a signal-blocking context only when initializing FireOpalSampler.

```python
from contextlib import contextmanager
import signal

@contextmanager
def block_signals():
    sigs = {signal.SIGUSR2}
    signal.pthread_sigmask(signal.SIG_BLOCK, sigs)
    try:
        yield
    finally:
        signal.pthread_sigmask(signal.SIG_UNBLOCK, sigs)

with block_signals():
    sampler = FireOpalSampler(mode=service.backend("ibm_sqc"))
```

This notebook is a translated version of the original documentation.
